# Semaine 7 : l'approche d'apprentissage de Bertsimas & Perakis

Cours du 17 novembre (`W7/Dynamic Pricing - A learning approach using nonlinear and linear optimization.pdf`) :
1. estimer **notre** demande `d1 = β0 + β1·p1 + β2·p2` à partir des données observées (LAD, β qui changent lentement : |β_t − β_t+1| ≤ δ, programme linéaire) ;
2. prendre la moyenne des β des N derniers jours pour prévoir demain ;
3. estimer le **comportement du concurrent**, en supposant qu'il choisit lui aussi un prix optimal qui dépend du nôtre : `p2 = (β0₂ + β1₂·p1) / (−2·β2₂)` ;
4. choisir notre prix en tenant compte de cette réaction : `max p1·(β0 + β1·p1 + β2·p2(p1))`.

Le code du prof (`W7/duopoly.py mit Pyomo File.txt`) utilise Pyomo + glpk + ipopt. **Sur DPC, Pyomo et ces solveurs ne sont probablement pas installés** : notre version utilise uniquement numpy / scipy.

In [ ]:
import os
os.chdir('..')   # racine du dépôt

# 0. Les nouveaux runs DPC (W4, 4/11 → 16/11) et la réaction des concurrents

In [ ]:
%run W7-Notebooks/w7_dpc_runs.py

- W4 : classement **8e à 13e** sur les 6 runs (11e, 9e, 8e, 13e, 12e, 11e), 47-63 % de 1res places, 0 % d'erreurs.
- **Les concurrents réagissent à notre prix** : la variation de leur prix aujourd'hui dépend de notre changement de prix d'hier.

In [ ]:
%run W7-Notebooks/w7_competitor_reaction.py

Sur 217 compétitions : 102 concurrents indépendants, 41 suiveurs faibles, 55 suiveurs forts et **19 copieurs exacts** (prix d'aujourd'hui = notre prix d'hier).
Contrôle « placebo » : notre changement de prix de *demain* explique beaucoup moins (91 compétitions significatives contre 130) : la réaction est réelle.
→ les étapes 3-4 du cours (anticiper le prix du concurrent) s'appliquent vraiment à la DPC.

# 1. Étapes 1-2 : apprendre la demande de la compétition en cours

Comparaison sur les vraies données (18 compétitions jamais vues, prédiction de la demande du jour t à partir du passé) :
- méthode du cours (LAD lisse, saison en cours, `scipy.optimize.linprog` / HiGHS) ;
- notre adaptation : partir du modèle commun (.pkl) et le corriger avec les saisons précédentes de la **même** compétition
  (l'`information_dump` est gardé d'une saison à l'autre). « niveau » = on corrige seulement a ; « a, b, c » = régression ridge vers le modèle commun.

In [ ]:
%run W7-Notebooks/w7_demand_learning.py

| Modèle | MSE | MAE |
|---|---|---|
| a, b, c appris | **0,604** | 0,634 |
| niveau appris | 0,614 | 0,641 |
| commun (W4) | 0,660 | 0,673 |
| cours (LAD lisse) | 0,762 | **0,603** |

Apprendre la compétition réduit l'erreur d'environ 8 %. La LAD du cours a la plus petite erreur absolue mais la plus grande erreur au carré :
comme en W6, avec 60 % de jours à 0 vente, LAD vise la médiane et sous-estime la demande.

# 2. Étapes 3-4 : anticiper la réaction du concurrent

Le prof suppose que le concurrent fixe un prix optimal qui dépend du nôtre *le même jour*. Sur DPC, les prix sont fixés en même temps :
le concurrent ne peut réagir qu'à notre prix **d'hier**. Notre modèle (appris pendant la compétition, moindres carrés sur 2 nombres) :

`prix_conc(t) − prix_conc(t−1) = α + k·(notre_prix(t−1) − prix_conc(t−1))`, avec k entre 0 (indépendant) et 1 (copieur).

Étape 4 : si on change notre prix de Δ, le concurrent aura suivi en moyenne une part γ de Δ sur les 20 prochains jours
(γ ≈ 0,95 pour un copieur, ≈ 0,5 pour k = 0,1). La demande devient `a + b·p + c·(pc + γ·(p − dernier prix))`,
donc la pente devient `b + c·γ` (moins négative) : **baisser le prix rapporte moins de clients, car le concurrent baisse aussi** → le prix optimal monte.
Le même calcul est utilisé dans le plan de prix sous contrainte de capacité de la semaine 6.

# 3. Simulation

Nouveau simulateur (`w7_simulation.py`) : 259 compétitions (toutes celles de Simon et d'Alois), 30 saisons jouées **à la suite** (l'algorithme apprend),
deux types de concurrent :
- **réactif** : il suit notre prix avec le k mesuré dans les données de cette compétition (optimiste pour la réaction : c'est le modèle que l'algorithme suppose) ;
- **rejoué** : prix rejoués tels quels, il ne réagit jamais (pessimiste pour la réaction).

In [ ]:
# environ 10 minutes (14 simulations en parallèle)
%run W7-Notebooks/w7_simulation.py

Revenu moyen par saison :

| Version | Concurrent réactif | Concurrent rejoué |
|---|---|---|
| W4 (sur DPC) | 2165 | 2071 |
| W6 (optimisé sous contrainte) | 2189 | 2091 |
| W7 demande apprise (niveau) | 2292 (+6 %) | 2156 (+4 %) |
| W7 demande apprise (a, b, c) | 2178 | 2182 |
| W7 réaction seule | 2625 | 1960 |
| **W7 niveau + réaction (choisie)** | **2715 (+25 %)** | 2011 (−3 %) |
| W7 niveau + réaction, sans marges | 3453 (+59 %) | 1583 (−24 %) |

- Apprendre le niveau de demande : gain dans les deux simulateurs (le plus sûr).
- Anticiper la réaction : très gros gain si le concurrent suit vraiment, petite perte sinon. Dans le simulateur « rejoué », la perte vient
  surtout des concurrents suiveurs (qui là ne suivent plus) ; face aux concurrents indépendants, la perte n'est que d'environ 1 %.
- Sans les marges : gain énorme ou perte énorme selon le simulateur → trop risqué pour l'instant.
- Version déposée : **W6 + niveau appris + réaction, avec marges** (`USE_ONLINE_DEMAND = True`, `ONLINE_MODE = 'level'`, `USE_REACTION = True`, `FOLLOW_BAND = True`).
- Limite : on simule notre revenu, pas celui du concurrent. Le classement DPC compare les deux ; face à un copieur, des prix plus hauts font monter les deux revenus.